# Train Bi-Encoder on A100 GPU

**IMPORTANT: Set Runtime to GPU (A100)**
- Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: A100

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository from GitHub
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Verify data exists
!ls -lh data/processed/problems_train2000.jsonl
!echo "Standards will be downloaded from HuggingFace using --from_hf flag"

In [ ]:
# 5. Run training (THIS IS THE MAIN STEP - ~15 mins on A100)
!python scripts/train_biencoder.py \
  --problems_file data/processed/problems_train2000.jsonl \
  --from_hf \
  --max_steps 5000 \
  --batch_size 64 \
  --num_negatives 8 \
  --temperature 0.03 \
  --early_stopping_patience 0 \
  --fp16 \
  --output_dir outputs/biencoder_5k_gpu

In [ ]:
# 6. Verify training completed
!ls -lh outputs/biencoder_5k_gpu/checkpoint/

In [ ]:
# 7. Download trained model
import shutil
from google.colab import files

# Zip the checkpoint
shutil.make_archive('biencoder_5k_gpu', 'zip', 'outputs/biencoder_5k_gpu/checkpoint')

# Download
files.download('biencoder_5k_gpu.zip')

print("\n✅ Download complete! Extract this on your Mac and replace outputs/biencoder/checkpoint/")

## Next Steps (on your Mac)

1. Extract `biencoder_5k_gpu.zip`
2. Replace `outputs/biencoder/checkpoint/` with the extracted files
3. Run hybrid inference:
```bash
python scripts/run_hybrid_rag.py \
  --biencoder_path outputs/biencoder \
  --verifier_path outputs/verifier \
  --problems_file data/processed/problems_test_full.jsonl \
  --output_file outputs/hybrid_rag_predictions_new.jsonl \
  --top_k 20 \
  --verification_threshold 0.2
```
4. Evaluate:
```bash
python scripts/evaluate_predictions.py \
  --predictions_file outputs/hybrid_rag_predictions_new.jsonl \
  --problems_file data/processed/problems_test_full.jsonl
```